# Pharmaceutical Bottle Dataset Exploration

Welcome to the **Pharmaceutical Bottle Anomaly Detection** dataset exploration notebook!

In this notebook, we inspect the **MVTec AD Bottle** dataset layout, analyze the image counts across normal training data and defective test categories, display sample bottle images alongside ground-truth defect masks, and visualize category distributions with bar charts.

> **Key Principle of Unsupervised Anomaly Detection:**
> The training set (`train/good`) contains only **normal, healthy bottles**. The model (PatchCore) learns what a *normal* bottle looks like. During evaluation, it inspects test bottles (`good`, `broken_large`, `broken_small`, `contamination`) to identify and localize anomalies.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import pandas as pd

# Define project root
PROJECT_ROOT = Path("..").resolve()
DATASET_ROOT = PROJECT_ROOT / "mvtec_dataset" / "bottle"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Dataset Directory: {DATASET_ROOT}")

## 1. Directory Structure Analysis

Let's inspect the folder tree of `mvtec_dataset/bottle/`.

In [ ]:
def print_dir_tree(start_dir: Path, max_depth: int = 3, current_depth: int = 0):
    if current_depth > max_depth:
        return
    indent = "  " * current_depth
    print(f"{indent}├── {start_dir.name}/")
    for item in sorted(start_dir.iterdir()):
        if item.is_dir() and not item.name.startswith("."):
            print_dir_tree(item, max_depth, current_depth + 1)

print_dir_tree(DATASET_ROOT)

## 2. Image Counts per Category

Now let's count the total number of images in `train/good` and each `test` defect category.

In [ ]:
categories_data = []

# Train count
train_good_imgs = list((DATASET_ROOT / "train" / "good").glob("*.png"))
categories_data.append({"Split": "Train", "Category": "good (normal)", "Count": len(train_good_imgs)})

# Test counts
test_dir = DATASET_ROOT / "test"
for cat_folder in sorted(test_dir.iterdir()):
    if cat_folder.is_dir():
        imgs = list(cat_folder.glob("*.png"))
        categories_data.append({"Split": "Test", "Category": cat_folder.name, "Count": len(imgs)})

df_counts = pd.DataFrame(categories_data)
print("=== DATASET IMAGE COUNTS SUMMARY ===")
display(df_counts)

## 3. Class Distribution Bar Chart

Let's visualize the dataset counts in a bar chart.

In [ ]:
plt.figure(figsize=(10, 5), dpi=150)
colors = ["#20bf6b", "#4b7bec", "#eb3b5a", "#fa8231", "#8854d0"]
bars = plt.bar(df_counts["Split"] + "/" + df_counts["Category"], df_counts["Count"], color=colors[:len(df_counts)])
plt.title("MVTec AD Bottle Dataset Category Distribution", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Split / Category", fontsize=12, fontweight="bold")
plt.ylabel("Number of Images", fontsize=12, fontweight="bold")
plt.xticks(rotation=20, ha="right")

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval}", ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

## 4. Example Images & Ground Truth Masks

Let's display sample bottle images from normal training, normal test, and defect categories alongside their binary ground truth defect localization masks.

In [ ]:
sample_categories = ["good", "broken_large", "broken_small", "contamination"]
fig, axes = plt.subplots(len(sample_categories), 2, figsize=(8, 12), dpi=150)

for i, cat in enumerate(sample_categories):
    img_path = list((DATASET_ROOT / "test" / cat).glob("*.png"))[0]
    gt_mask_path = DATASET_ROOT / "ground_truth" / cat / f"{img_path.stem}_mask.png"
    
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f"Test Image: {cat}", fontsize=11, fontweight="bold")
    axes[i, 0].axis("off")
    
    if gt_mask_path.exists():
        mask = cv2.imread(str(gt_mask_path), cv2.IMREAD_GRAYSCALE)
        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title(f"Ground Truth Mask ({cat})", fontsize=11, fontweight="bold")
    else:
        axes[i, 1].text(0.5, 0.5, "No Defect Mask\n(Normal Bottle)", ha="center", va="center", fontsize=11, fontweight="bold")
        axes[i, 1].set_title(f"Ground Truth Mask ({cat})", fontsize=11, fontweight="bold")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()